# 第 7 章 · 多 agent（Sequential Handoffs）

两个 agent 排成流水线：前一个把结果写进 `session.state`，后一个从 state 里读。

`SequentialAgent` 本身不做推理，它只负责**按顺序调用**。数据不是靠返回值传的，是靠 **state 的键**。

kernel 选 `Python (agentic-adk)`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async, require_adk)
quiet_adk_sync_warnings()

require_adk()      # 【适配】内核选错时给出可操作提示（内部就是 import google.adk）

from google.adk.agents import LlmAgent, SequentialAgent

# ★ 本章新增 output_key：这个 agent 的最终答复会写进 state[name]
# 【适配】书里两个 agent 都没给 model / instruction，构造出来跑不了
step1 = LlmAgent(
    name="Step1_Fetch",
    model=adk_model(),
    instruction="给出三条关于用户主题的简短事实，每行一条。",
    output_key="data",
)

# 【适配】书里写 state['data']；ADK 只替换 {data} 占位符，那样写模型看不到数据
step2 = LlmAgent(
    name="Step2_Process",
    model=adk_model(),
    instruction="下面是上一步查到的资料：\n{data}\n\n请用一段话总结它。",
)

pipeline = SequentialAgent(name="MyPipeline", sub_agents=[step1, step2])

finals, state = run_once(pipeline, "量子计算")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:400]}")
print("\n--- state['data']（第 1 步的产物）---")
print((state.get("data") or "(空)")[:300])


--- [Step1_Fetch] ---
1. 量子计算的基本信息单位是量子比特（Qubit），它利用量子叠加和量子纠缠原理进行信息处理。
2. 相比传统计算机的二进制限制，量子计算在处理特定复杂问题（如大数分解、分子模拟）时具有指数级的加速潜力。
3. 目前全球量子计算正处于从“物理理论验证”向“工程化及实用化”过渡的发展阶段。

--- [Step2_Process] ---
量子计算以量子比特（Qubit）为基本信息单位，利用量子叠加和量子纠缠原理进行信息处理，相比传统二进制计算机，它在处理大数分解、分子模拟等特定复杂问题时展现出指数级的加速潜力，目前全球该技术正处于从“物理理论验证”向“工程化及实用化”迈进的关键过渡阶段。

--- state['data']（第 1 步的产物）---
1. 量子计算的基本信息单位是量子比特（Qubit），它利用量子叠加和量子纠缠原理进行信息处理。
2. 相比传统计算机的二进制限制，量子计算在处理特定复杂问题（如大数分解、分子模拟）时具有指数级的加速潜力。
3. 目前全球量子计算正处于从“物理理论验证”向“工程化及实用化”过渡的发展阶段。
